# Route 1 — MC-GPU v1.3_PCD Colab baseline
Run all cells after selecting a GPU runtime.  The first Drive mount requires OAuth. P1 and P2 write state and raise on failure; P3 refuses to run unless the recorded 3σ physical gate passed. No MC-GPU source is modified (2026-08-04: pinned to DIDSR/MCGPUv1.3_PCD, the sibling repo that actually ships sample data — see plan addendum 2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, hashlib, math, os, pathlib, re, shutil, subprocess, sys, time, statistics
from datetime import datetime, timezone
ROOT = pathlib.Path('/content/route1_mcgpu'); REPO = ROOT/'MCGPU'; OUT = pathlib.Path('/content/drive/MyDrive/viveMonte/route1_mcgpu')
ROOT.mkdir(exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
STATE = ROOT/'state.json'; LOG = OUT/'execution.log'
# 2026-08-04: switched from DIDSR/MCGPU (source-only, no sample data at all -- see plan
# addendum) to DIDSR/MCGPUv1.3_PCD, an FDA/DIDSR sibling repo carrying the same v1.3
# lineage (CC0) plus real, verified pencil/fan-beam sample inputs and real water/air
# PENELOPE-derived material files. Confirmed by reading source directly: Nbin<=0 in the
# IMAGE DETECTOR section falls back to plain v1.3 Energy-Integrated-Detector behaviour,
# so the PCD-specific energy-binned output structure is never exercised here.
COMMIT = 'af5fa2888ebbc71ed3aaaf73e8ea8c1b24aea846'  # DIDSR/MCGPUv1.3_PCD main, pinned 2026-08-04
URL = 'https://github.com/DIDSR/MCGPUv1.3_PCD.git'
EXE = 'MC-GPU_v1.3_PCD.x'; SRC = 'MC-GPU_v1.3_PCD.cu'
EGS5_RATE_1 = 1_520_000.0; EGS5_RATE_8 = 8_800_000.0; EGS5_LCG_RATE = 2_800_000.0
def run(cmd, cwd=None, check=True):
    p = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    LOG.open('a').write('$ '+ ' '.join(map(str, cmd))+'\n'+p.stdout+'\n')
    if check and p.returncode: raise RuntimeError(p.stdout)
    return p
def sha(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
def save_state(**kw):
    old = json.loads(STATE.read_text()) if STATE.exists() else {}
    old.update(kw); STATE.write_text(json.dumps(old, indent=2)); (OUT/'state.json').write_text(json.dumps(old, indent=2))
def require(key, value=True):
    s=json.loads(STATE.read_text()) if STATE.exists() else {}
    if s.get(key) != value: raise RuntimeError(f'BLOCKED: requires {key}={value}; recorded state is {s.get(key)!r}')
LOG.write_text(f'route1 started {datetime.now(timezone.utc).isoformat()}\n')

## P1 — fixed acquisition, no-source-change build decision

In [ ]:
# Fixed commit only: no branch or latest-HEAD acquisition.
if REPO.exists(): shutil.rmtree(REPO)
run(['git','clone',URL,str(REPO)])
run(['git','checkout','--detach',COMMIT], REPO)
assert run(['git','rev-parse','HEAD'], REPO).stdout.strip() == COMMIT
env = {
  'nvcc_version': run(['nvcc','--version']).stdout,
  'nvidia_smi': run(['nvidia-smi','--query-gpu=name,driver_version,compute_cap','--format=csv,noheader']).stdout,
  'commit': COMMIT, 'repo_url': URL,
}
# The sole permitted build-setting substitution is sm_75; source hashes prove no source edit.
# Base flags mirror the repo's own Makefile (CFLAGS), dropping the openmpi/-DUSING_MPI
# bits (single-GPU, no MPI needed) and the repo's own -I paths that assume a local CUDA
# Samples SDK install (not present on Colab -- see the missing-header fallback below).
source_before={p.name:sha(p) for p in REPO.glob('*.cu')} | {p.name:sha(p) for p in REPO.glob('*.h')}
build=['nvcc','-m64','-DUSING_CUDA',SRC,'-o',EXE,'-O3','-use_fast_math','-I.','-lz','--ptxas-options=-v','-arch=sm_75']
p=run(build, REPO, check=False); source_after={p.name:sha(p) for p in REPO.glob('*.cu')} | {p.name:sha(p) for p in REPO.glob('*.h')}
env['build_command']=' '.join(build); env['source_unchanged']=(source_before==source_after); env['build_returncode']=p.returncode
env['first_attempt_build_log']=p.stdout
cuda_samples_used=False
# Empirically observed (Colab, CUDA 12.8, T4, 2026-08-04, on the sibling DIDSR/MCGPU v1.3
# repo -- same underlying source lineage, same header): MC-GPU_v1.3*.h expects
# helper_functions.h, a vendored NVIDIA CUDA Samples header, not MC-GPU's own code.
# CUDA Samples were bundled with the Toolkit circa 2012-2018 but are now a separate
# NVIDIA/cuda-samples repo not installed by default. Adding a missing third-party header
# search path via an extra -I is a build-setting addition, not a source edit to MC-GPU --
# the unchanged-source-hash check below still applies and still gates this.
if p.returncode and 'helper_functions.h' in (p.stdout or ''):
    cuda_samples_used=True
    CUDA_SAMPLES = ROOT/'cuda-samples'
    if CUDA_SAMPLES.exists(): shutil.rmtree(CUDA_SAMPLES)
    # No commit is pre-registered for this repo (the dependency itself was not foreseen
    # by the plan); the commit actually fetched is recorded below rather than assumed.
    run(['git','clone','--depth','1','https://github.com/NVIDIA/cuda-samples.git',str(CUDA_SAMPLES)])
    env['cuda_samples_commit']=run(['git','rev-parse','HEAD'], CUDA_SAMPLES).stdout.strip()
    env['cuda_samples_url']='https://github.com/NVIDIA/cuda-samples.git'
    common_inc = CUDA_SAMPLES/'Common'
    build2 = build[:-1] + [f'-I{common_inc}', build[-1]]
    p=run(build2, REPO, check=False); source_after={p.name:sha(p) for p in REPO.glob('*.cu')} | {p.name:sha(p) for p in REPO.glob('*.h')}
    env['build_command']=' '.join(build2); env['source_unchanged']=(source_before==source_after); env['build_returncode']=p.returncode
    env['second_attempt_build_log']=p.stdout
env['cuda_samples_used_as_missing_header_fallback']=cuda_samples_used
if p.returncode or not env['source_unchanged']:
    save_state(p1_pass=False, p1=env, terminal_reason='P1 failed: sm_75-only build did not succeed (with or without the missing-header fallback); source modification would be required or source changed.')
    raise SystemExit('P1 FAILED. Recorded to Drive. Do not execute P2/P3/P4; this route ends here.')
save_state(p1_pass=True, p1=env)
print('P1 build passed with -arch=sm_75 and unchanged MC-GPU source' + (' (required the cuda-samples header fallback).' if cuda_samples_used else '.'))

In [ ]:
# P1 specification record. Unlike the previous route1 revision, these facts are now
# confirmed by reading the actual repository (README.md + MC-GPU_v1.3_PCD.cu source,
# 2026-08-04 offline research), not assumed. Kept here as the audit trail; also save
# the raw README for anyone re-checking later.
require('p1_pass')
readme=(REPO/'README.md').read_text(errors='replace')
spec={
 'input_format':'MC-GPU .in sections (v1.3 syntax; PCD adds one extra line to the IMAGE DETECTOR section: Emin Emax Nbin)',
 'spectrum_format':'two columns per line: lower-edge energy in eV, relative probability; terminated by a line with a negative probability (verified in init_energy_spectrum()). A single exact-energy bin is expressed as two identical-energy lines, e.g. "60000 1.0" then "60000 -1" (zero-width bin -> energy sampled exactly, verified against the Walker-alias interpolation code).',
 'geometry':'penEasy-derived voxel format, [SECTION VOXELS HEADER v.2008-04-13]; X fastest, then Y, then Z; one blank line after each completed X-row and one further blank line after each completed Z-slice when the header flags blank-line-separated cycles (verified against the shipped gammexPhantom.vox: 100x100x1 voxels, 10000 data lines + 101 blank lines).',
 'source_shape':'point source, rectangular collimated cone beam; POLAR AND AZIMUTHAL APERTURES = 0 0 gives a true zero-divergence pencil beam (verified against the shipped Sample_Pencil_Beam/pencil_beam_simulation.in); negative apertures mean "cover the whole detector", not narrow (this was a misreading in route1 revision 2 -- corrected here).',
 'primary_tally':'5 consecutive per-pixel float32 blocks in the .raw binary output, order = [total(scatter+primary), non-scattered(primary), Compton, Rayleigh, multi-scatter] -- verified directly in report_image()\'s Nbin<=0 (Energy-Integrated Detector) branch. Values are ALREADY normalized to eV/cm^2 per history (NORM includes 1/total_histories) -- route1 revision 2 divided by N a second time, a real double-normalization bug fixed here.',
 'pcd_vs_eid':'Nbin<=0 in the IMAGE DETECTOR Emin/Emax/Nbin line selects plain v1.3 Energy-Integrated-Detector behaviour (verified in source comments and the image_bytes allocation branch); this route always uses Nbin<=0 so the PCD-specific energy-binned array layout is never exercised.',
 'output_path':'{cwd}/output/EID/{file_name_output} (ASCII) and {cwd}/output/EID/{file_name_output}.raw (binary), auto-created by the program itself -- verified in report_image(); NOT {cwd}/{file_name_output} as assumed in route1 revision 2.',
 'rng':'RANECU; per-thread sequences initialized with seedsMLCG',
 'material_source':'PENELOPE-2006-derived .mcgpu files; the base DIDSR/MCGPU v1.3 repo ships NO material or sample data at all (source-only). Real water/air material files exist in the sibling DIDSR/MCGPUv1.3_PCD repo at Sample_Fan_Beam/inputs/{water,air}.mcgpu and are copied into this route\'s own scenario unmodified (SHA-256 recorded in P2).',
}
(OUT/'P1_specification.txt').write_text(readme)
save_state(p1_spec=spec)
print(json.dumps(spec, indent=2))

In [ ]:
# P1 completion: the pinned package's shipped sample must execute against its own
# shipped data (real water/air materials, real spectrum, real voxel phantom) before P2
# is allowed. The base DIDSR/MCGPU v1.3 repo had no such data at all (route1 revision 1
# failure); this fork (DIDSR/MCGPUv1.3_PCD) ships Sample_Fan_Beam/ complete and working.
# Only the histories count and CT projection count are reduced (via regex on the shipped
# .in text) so the smoke test finishes quickly; every referenced data file (spectrum,
# voxel phantom, all four materials) is used completely unmodified.
require('p1_pass')
sample_dir = REPO/'Sample_Fan_Beam'
sample_in_path = sample_dir/'fan_beam_simulation.in'
original_sample_in = sample_in_path.read_text()
reduced = re.sub(r'^\S+(?=\s+# TOTAL NUMBER OF HISTORIES)', '1e5', original_sample_in, count=1, flags=re.M)
reduced = re.sub(r'^\S+(?=\s+# NUMBER OF PROJECTIONS)', '1', reduced, count=1, flags=re.M)
assert reduced != original_sample_in, 'expected substitutions did not match the shipped .in file; format may have changed upstream'
smoke_in = sample_dir/'fan_beam_simulation_p1_smoke.in'
smoke_in.write_text(reduced)
sample=run([str((REPO/EXE).resolve()), smoke_in.name], sample_dir, check=False)
if sample.returncode:
    save_state(p1_pass=False, terminal_reason='P1 failed: pinned distribution sample input did not execute.', p1_sample_log=sample.stdout)
    raise SystemExit('P1 FAILED: sample input did not execute. P2/P3/P4 are blocked.')
save_state(p1_sample_pass=True)
print('P1 sample execution passed (reduced N=1e5, 1 projection, real shipped data unmodified).')

## P2 — fixed water/air controls and 3σ gate
The input creator writes the 10 cm voxel slab, a zero-width (exact) 60 keV spectrum bin, and a true zero-divergence pencil beam (aperture 0 0, verified against the shipped Sample_Pencil_Beam example) centred on the one-pixel detector. The air-only conversion control is mandatory because image values are eV/cm² per history.

In [ ]:
import numpy as np
# Verified against report_image() (Nbin<=0 / Energy-Integrated-Detector branch):
# output is written to {cwd}/output/EID/{file_name_output} (ASCII) and
# {cwd}/output/EID/{file_name_output}.raw (binary), NOT next to the input file as
# route1 revision 2 assumed. The binary holds 5 consecutive pixels_per_image blocks:
# [total(scatter+primary), non-scattered(primary), Compton, Rayleigh, multi-scatter],
# each value already normalized to eV/cm^2 PER HISTORY (the NORM constant in the source
# divides by total_histories internally) -- do not divide by N again downstream.
def read_eid_image(image_file_name):
    p = REPO/'output'/'EID'/f'{image_file_name}.raw'
    return np.fromfile(p, dtype=np.float32)


In [ ]:
require('p1_pass'); require('p1_sample_pass')
# Real, verified inputs only -- route1 revision 2 fabricated material filenames and a
# voxel-header keyword that don't exist anywhere in any MC-GPU distribution; both are
# replaced here with values confirmed against the shipped Sample_Fan_Beam data and the
# report_image()/init_energy_spectrum() source (see P1 spec cell above for citations).
WATER_MCGPU = REPO/'Sample_Fan_Beam'/'inputs'/'water.mcgpu'
AIR_MCGPU = REPO/'Sample_Fan_Beam'/'inputs'/'air.mcgpu'
assert WATER_MCGPU.exists() and AIR_MCGPU.exists(), 'shipped water/air material files missing -- upstream layout may have changed'
shutil.copy2(WATER_MCGPU, REPO/'water.mcgpu'); shutil.copy2(AIR_MCGPU, REPO/'air.mcgpu')

def write_voxels(path, material, density, nx=10, ny=10, nz=10):
    # penEasy-derived format verified against the shipped gammexPhantom.vox: X fastest,
    # then Y, then Z; one blank line after each completed X-row, one further blank line
    # after each completed Z-slice (matches the 100x100x1 example: 10000 data lines +
    # 101 blank lines = 100 X-row blanks + 1 Z-slice blank).
    lines=['# HEADER section: homogeneous slab generated for route1_mcgpu_colab.ipynb',
           '[SECTION VOXELS HEADER v.2008-04-13]', f'{nx} {ny} {nz}', '1.0 1.0 1.0',
           ' 1', ' 2', ' 1', '[END OF VXH SECTION]']
    for _z in range(nz):
        for _y in range(ny):
            for _x in range(nx):
                lines.append(f'{material} {density:.7g}')
            lines.append('')
        lines.append('')
    pathlib.Path(path).write_text('\n'.join(lines)+'\n')

def write_spectrum(path, energy_eV):
    # Verified against init_energy_spectrum(): a zero-width bin (identical energy on the
    # bin line and its negative-probability terminator) samples that exact energy every
    # time via the Walker-alias linear interpolation, i.e. a true monoenergetic source.
    pathlib.Path(path).write_text(f'{energy_eV:g} 1.0\n{energy_eV:g} -1\n')

def write_input(path, histories, voxel_file, image_file, material_file):
    pathlib.Path(path).write_text(f'''#[SECTION SIMULATION CONFIG v.2009-05-12]
{histories}
1234567890
1
128
150
#[SECTION SOURCE v.2011-07-12]
mono60keV.spc
5.0 -20.0 5.0
0.0 1.0 0.0
0 0
#[SECTION IMAGE DETECTOR v.2009-12-02]
{image_file}
1 1
1.0 1.0
1000 200000 0
40.0
#[SECTION CT SCAN TRAJECTORY v.2011-10-25]
1
0.0
0.0 5000.0
0.0
0.0
#[SECTION DOSE DEPOSITION v.2012-12-12]
No
No
unused_dose.dat
1 1
1 1
1 1
#[SECTION VOXELIZED GEOMETRY FILE v.2009-11-30]
{voxel_file}
#[SECTION MATERIAL FILE LIST v.2009-11-30]
{material_file}
''')

write_spectrum(REPO/'mono60keV.spc', 60000)
write_voxels(REPO/'air10cm.vox', 1, 0.00120479); write_voxels(REPO/'water10cm.vox', 1, 1.0)
write_input(REPO/'air_control.in', 1_000_000, 'air10cm.vox', 'air_image.dat', 'air.mcgpu')
write_input(REPO/'water_gate.in', 10_000_000, 'water10cm.vox', 'water_image.dat', 'water.mcgpu')
files=['mono60keV.spc','air10cm.vox','water10cm.vox','air_control.in','water_gate.in','water.mcgpu','air.mcgpu']
manifest={f:sha(REPO/f) for f in files}; (OUT/'sha256_manifest.json').write_text(json.dumps(manifest,indent=2))
save_state(p2_inputs=manifest, aperture_note='0 0 = exact zero-divergence pencil beam, verified against Sample_Pencil_Beam; no geometric path-length error budget needed (route1 revision 2 assumed a cone beam and built an error budget for a misread aperture sign -- removed).')

In [ ]:
require('p1_pass')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'xraylib'])
# Run unmodified executable and read the 5 float32 blocks; primary is block index 1
# (pixels_per_image=1 here, so blocks and pixel-0 offsets coincide).
def execute(inp, label):
    t0=time.perf_counter(); p=run([str((REPO/EXE).resolve()),inp], REPO, check=False); elapsed=time.perf_counter()-t0
    if p.returncode: raise RuntimeError(f'{label} failed:\n{p.stdout}')
    # Verified against a real execution.log (2026-08-05): MC-GPU's own performance report reads
    # ">>> Time spent in the Monte Carlo transport only: 0.021 s." -- this is the GPU-transport-only
    # host-side timer MC-GPU itself brackets around the CUDA kernel launch, isolating it from
    # initialization/report/cleanup ("Time spent in initialization, reporting and clean up") and
    # from the coarser 2-decimal "Simulation time [s]" line in the earlier IMAGE TALLY block. An
    # earlier notebook revision's regex assumed a "GPU...time...Xs" pattern that never occurs in
    # real output and silently never matched.
    kernel=re.search(r'Time spent in the Monte Carlo transport only:\s*([0-9.]+)\s*s',p.stdout)
    return elapsed, (float(kernel.group(1)) if kernel else None), p.stdout
air_e2e, air_kernel, air_log=execute('air_control.in','air control')
air_raw=read_eid_image('air_image.dat'); assert air_raw.size >= 5
# eV/cm^2/history -> transmission probability: value * pixel_area_cm2 / photon_energy_eV.
# No division by N: report_image()'s NORM already divides by total_histories (verified
# in source; route1 revision 2 divided by N a second time, a real bug, fixed here).
pixel_area=1.0
air_T=float(air_raw[1])*pixel_area/60_000.0; air_se=math.sqrt(max(air_T*(1-air_T)/1_000_000,0.0))
import xraylib
# The control is NOT "air is transparent" (route1 revision 3 wrongly asserted T~=1.0 and
# failed a real run at T=0.99771): air's *total* interaction cross section at 60 keV is
# dominated by incoherent (Compton) scattering, not absorption, so xraylib CS_Total gives
# mu/rho=0.1875 cm^2/g -- over a 10 cm, 0.00120479 g/cm^3 path that is a genuine ~0.226%
# non-scattered-fraction loss (T_analytic=0.99774), not noise. The control must compare
# MC-GPU's air transmission against this analytic Beer-Lambert value, exactly like the
# water gate below, not against a hardcoded 1.0.
mu_air=xraylib.CS_Total_CP('Air, Dry (near sea level)',60.0); air_analytic=math.exp(-mu_air*0.00120479*10.0)
assert abs(air_T-air_analytic) <= 3*max(air_se,1e-6), f'air conversion control failed: MC-GPU T={air_T}, analytic={air_analytic}'
water_e2e, water_kernel, water_log=execute('water_gate.in','water gate')
water_raw=read_eid_image('water_image.dat')
mc_t=float(water_raw[1])*pixel_area/60_000.0; mc_se=math.sqrt(mc_t*(1-mc_t)/10_000_000)
mu=xraylib.CS_Total_CP('Water, Liquid',60.0); analytic=math.exp(-mu*10.0)
# Existing EGS5 water60_bound result (IBOUND=1, bound Compton; the pre-registered-tolerance-passing run, not the superseded free-electron IBOUND=0 run -- see docs/egs5_crosscheck/RESULTS.md, '第2回実行'): 0.1270 +/- 0.00047 (500,000 histories).
egs5_t, egs5_se = 0.1270, 0.00047
# Aperture is an exact 0 0 pencil beam (verified), so there is no geometric divergence
# path-length term to add to the comparison budget (route1 revision 2 assumed a cone
# beam and built one for a misread aperture-sign convention -- removed).
def sigma_delta(a,sa,b,sb): return abs(a-b)/math.sqrt(sa*sa+sb*sb)
z_analytic=sigma_delta(mc_t,mc_se,analytic,0.0); z_egs5=sigma_delta(mc_t,mc_se,egs5_t,egs5_se)
# 2026-08-05: a real Colab run failed the water gate at z_analytic=13.5σ while z_egs5 passed at
# 1.67σ. Root cause (verified against the actual shipped water.mcgpu, not assumed -- see below):
# MC-GPU's own PENELOPE-2006 total cross section for water at 60 keV differs from xraylib's
# EPDL-based CS_Total_CP by ~0.5% in mu, which compounds to ~1.1% in 10cm transmission; at
# N=1e7 MC-GPU's own statistical SE is tight enough (~0.08% relative) that this known
# inter-database difference alone registers as >3σ. This is a database-choice difference
# (pre-registered category 4: source shape / unit conversion / geometric discretization /
# cross-section data -- the other three were independently ruled out: pencil beam confirmed,
# air control passes cleanly, straight-through path length is geometrically exact), not a
# code or MC-GPU defect -- confirmed by an independent Codex review of both the diagnosis and
# this cell's code before this fix was written.
def read_material_mu(material_path, target_energy_eV):
    # Parses the real [MEAN FREE PATHS] table shipped in this run's own water.mcgpu/air.mcgpu
    # (PENELOPE-2006-derived, copied unmodified from Sample_Fan_Beam/inputs/) and linearly
    # interpolates the TOTAL column (mean free path in cm, at the file's own nominal density)
    # at target_energy_eV. Returns mu=1/mfp. This checks that MC-GPU's simulation is
    # self-consistent with the physics data it was given -- a different, narrower question
    # than whether an independent database (xraylib/EPDL) agrees with PENELOPE-2006 on the
    # underlying cross sections, which is a known, expected type of inter-code difference
    # (this project's own CLAUDE.md documents an analogous xraylib-vs-NIST divergence
    # elsewhere) and is recorded below as reference information only, not gated on.
    rows=[]
    with open(material_path) as f:
        for line in f:
            if line.startswith('#') or not line.strip(): continue
            parts=line.split()
            if len(parts) < 5: continue
            rows.append((float(parts[0]), float(parts[4])))
    rows.sort()
    for (e0,m0),(e1,m1) in zip(rows, rows[1:]):
        if e0 <= target_energy_eV <= e1:
            frac=(target_energy_eV-e0)/(e1-e0)
            return 1.0/(m0+frac*(m1-m0))
    raise ValueError(f'{target_energy_eV} eV outside material table range')
mu_water_penelope=read_material_mu(REPO/'water.mcgpu', 60000.0)
water_self_analytic=math.exp(-mu_water_penelope*10.0)
z_self=sigma_delta(mc_t,mc_se,water_self_analytic,0.0)
# Gate: EGS5 comparison (independent code, cross-validated scenario) AND self-consistency
# (this MC-GPU build correctly simulates the physics data it was shipped with). The xraylib
# comparison is recorded for transparency but is advisory, not gating, per the reasoning above
# (user-approved 2026-08-05, Codex-reviewed).
gate=(z_self <= 3.0 and z_egs5 <= 3.0)
record={'air_control':{'T':air_T,'se':air_se,'e2e_s':air_e2e,'analytic':air_analytic},'water':{'T':mc_t,'se':mc_se,'e2e_s':water_e2e,'kernel_s':water_kernel},'analytic_xraylib_reference_only':{'T':analytic,'mu_cm_inv':mu,'z':z_analytic,'note':'PENELOPE-2006 vs xraylib/EPDL cross-section database difference, not a code defect -- see comments above'},'self_consistency':{'mu_penelope_cm_inv':mu_water_penelope,'T':water_self_analytic,'z':z_self},'egs5':{'T':egs5_t,'se':egs5_se},'z_egs5':z_egs5,'pass':gate}
(OUT/'p2_gate.json').write_text(json.dumps(record,indent=2)); save_state(p2_pass=gate,p2=record)
if not gate: raise SystemExit('P2 PHYSICAL GATE FAILED (strict 3σ, EGS5 and self-consistency). P3 is blocked; investigate and record cause, do not measure throughput.')
print(json.dumps(record,indent=2))

## P3 — throughput (blocked unless P2 passed)
One warm-up plus three retained repeats are executed for each N. Kernel time is MC-GPU's own "Time spent in the Monte Carlo transport only" report line (verified against a real execution.log, 2026-08-05) when available; the parser fails rather than substituting wall time. End-to-end is process wall clock. The gap between them is reported as a single honest 'context_init_and_io_residual_s' (host I/O + CUDA context init + uninstrumentable host work) — MC-GPU is an unmodified black box, so this is not split further (an strace-based split was considered and dropped: strace's own overhead would bias the timing under measurement).

In [ ]:
require('p2_pass')
def timed_run(n, repeat, warmup=False):
    write_input(REPO/'bench.in', n, 'water10cm.vox', 'bench_image.dat', 'water.mcgpu')
    gpu_before=run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader']).stdout.strip()
    t0=time.perf_counter(); p=run([str((REPO/EXE).resolve()),'bench.in'],REPO,check=False); e2e=time.perf_counter()-t0
    if p.returncode: raise RuntimeError(p.stdout)
    m=re.search(r'Time spent in the Monte Carlo transport only:\s*([0-9.]+)\s*s',p.stdout)
    if not m: raise RuntimeError('No GPU-transport-only time found in unmodified MC-GPU output; do not replace it with wall time.')
    kernel=float(m.group(1)); gpu_after=run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader']).stdout.strip()
    raw=read_eid_image('bench_image.dat'); T=float(raw[1])*pixel_area/60_000.0; R=math.sqrt((1-T)/(n*T))
    # No strace-based read/write split: MC-GPU is an unmodified black box and strace overhead
    # would itself bias the very timing under measurement, so host I/O + context init are
    # reported as one honest residual rather than a fabricated split.
    residual_s = e2e - kernel
    # gpu_execution_confirmed is primarily evidenced by the parsed GPU-transport-only time above
    # (a CPU-fallback binary would emit no such line and 'm' would already have raised); the
    # nvidia-smi before/after snapshots are corroborating telemetry, not the sole proof.
    return {'N':n,'repeat':repeat,'warmup':warmup,'kernel_s':kernel,'kernel_histories_s':n/kernel,'end_to_end_s':e2e,'effective_histories_s':n/e2e,'context_init_and_io_residual_s':residual_s,'primary_T':T,'primary_R':R,'FOM':1/(R*R*e2e),'gpu_before':gpu_before,'gpu_after':gpu_after,'gpu_execution_confirmed':True}
rows=[]
for n in (1_000_000,10_000_000,100_000_000):
    timed_run(n,'warmup',True)
    rows.extend(timed_run(n,i) for i in range(1,4))
for r in rows:
    r['ratio_vs_egs5_O']=r['kernel_histories_s']/EGS5_RATE_1; r['ratio_vs_egs5_8proc']=r['kernel_histories_s']/EGS5_RATE_8; r['ratio_vs_egs5_lcg']=r['kernel_histories_s']/EGS5_LCG_RATE
summary={str(n):{'kernel_median':statistics.median([r['kernel_histories_s'] for r in rows if r['N']==n]),'kernel_range':[min(r['kernel_histories_s'] for r in rows if r['N']==n),max(r['kernel_histories_s'] for r in rows if r['N']==n)],'effective_median':statistics.median([r['effective_histories_s'] for r in rows if r['N']==n])} for n in (1_000_000,10_000_000,100_000_000)}
(OUT/'p3_measurements.json').write_text(json.dumps({'rows':rows,'summary':summary},indent=2)); save_state(p3_complete=True,p3_summary=summary)
print(json.dumps(summary,indent=2))

## P4 — durable record package
This final cell copies the generated inputs, logs, hashes, and structured results to Drive. Paste the values into the repository `RESULTS.md`, add the permitted speed-comparison cross-reference, then request the required vive-audit.

In [ ]:
require('p3_complete')
for p in [LOG, STATE, OUT/'sha256_manifest.json', OUT/'p2_gate.json', OUT/'p3_measurements.json']:
    assert p.exists(), p
for name in ['mono60keV.spc','air10cm.vox','water10cm.vox','air_control.in','water_gate.in','bench.in']:
    shutil.copy2(REPO/name, OUT/name)
artifact_manifest={p.name:sha(p) for p in OUT.iterdir() if p.is_file()}
(OUT/'artifact_sha256.json').write_text(json.dumps(artifact_manifest,indent=2))
save_state(p4_complete=True,artifact_manifest=artifact_manifest)
print('P4 complete:', OUT)
print('Next repository actions are deliberately manual: import the Drive artifact, complete RESULTS.md, append only a cross-reference to speed_comparison/RESULTS.md, then run vive-audit.')